In [1]:
import pandas as pd

df = pd.read_csv('/kaggle/input/datasets/nettrust/nas-drives-cmrsmr-failure-rates-and-specs-2026/drives-2026-06-17.csv')
print(f"{len(df)} drives | {df['brand'].nunique()} brands | {df['line'].nunique()} product lines")
df.head()

142 drives | 3 brands | 39 product lines


,model,brand,line,capacity_tb,rpm,cache_mb,interface,form_factor,recording_tech,acoustic_idle_db,acoustic_seek_db,is_helium,drive_class,in_production,afr_pct,reliability_drive_count,reliability_drive_days,reliability_failures,reliability_source
0,WUH722626ALE6L4,WD,Ultrastar DC HC590,26,7200,512,SATA,3.5,cmr,20.0,36.0,1.0,Enterprise,1,0.40,1200.0,91690.0,1.0,Backblaze 2025
1,ST24000NM002H,Seagate,Exos X24,24,7200,512,SATA,3.5,cmr,30.0,34.0,1.0,Enterprise,1,2.75,8328.0,1941010.0,146.0,Backblaze 2025
2,ST24000NT002,Seagate,IronWolf Pro,24,7200,512,SATA,3.5,cmr,28.0,32.0,1.0,NAS-Pro,1,NaN,NaN,NaN,NaN,NaN
3,ST24000VE002,Seagate,SkyHawk AI,24,7200,512,SATA,3.5,cmr,NaN,NaN,1.0,Surveillance,1,NaN,NaN,NaN,NaN,NaN
4,MG11ACA24TE,Toshiba,MG11,24,7200,512,SATA,3.5,cmr,20.0,NaN,1.0,Enterprise,1,0.68,4739.0,374087.0,7.0,Backblaze 2025


## CMR vs SMR breakdown

SMR drives use shingled recording to pack more capacity - but they stall under sustained writes of a RAID rebuild, which can drop them from the array mid-rebuild.

In [2]:
df['recording_tech'].value_counts()

recording_tech
cmr    124
smr     18
Name: count, dtype: int64

## SMR drives to avoid in a NAS/RAID array

In [3]:
df[df['recording_tech'] == 'smr'][['brand', 'line', 'model', 'capacity_tb']].sort_values(['brand', 'capacity_tb'])

,brand,line,model,capacity_tb
127,Seagate,BarraCuda,ST2000DM008,2
123,Seagate,BarraCuda,ST3000DM007,3
109,Seagate,BarraCuda,ST4000DM004,4
94,Seagate,BarraCuda,ST6000DM003,6
78,Seagate,BarraCuda,ST8000DM004,8
139,Toshiba,L200,HDWL110,1
140,Toshiba,MQ04,MQ04ABF100,1
113,Toshiba,DT02,DT02ABA400,4
114,Toshiba,P300,HDWD240,4
98,Toshiba,DT02,DT02ABA600,6


## Most reliable drives (Backblaze annualized failure rate, lowest first)

In [4]:
df[df['afr_pct'].notna()][['brand', 'line', 'model', 'capacity_tb', 'recording_tech', 'afr_pct', 'reliability_drive_count']].sort_values('afr_pct')

,brand,line,model,capacity_tb,recording_tech,afr_pct,reliability_drive_count
79,Seagate,Exos 7E8,ST8000NM000A,8,cmr,0.00,245.0
0,WD,Ultrastar DC HC590,WUH722626ALE6L4,26,cmr,0.40,1200.0
14,WD,Ultrastar DC HC570,WUH722222ALE6L4,22,cmr,0.47,44265.0
32,Seagate,Exos X16,ST16000NM001G,16,cmr,0.54,34305.0
18,Toshiba,MG10,MG10ACA20TE,20,cmr,0.59,17999.0
4,Toshiba,MG11,MG11ACA24TE,24,cmr,0.68,4739.0
42,WD,Ultrastar DC HC550,WUH721816ALE6Lx,16,cmr,0.80,29629.0
54,WD,Ultrastar DC HC530,WUH721414ALE6L4,14,cmr,0.81,8630.0
38,Toshiba,MG08,MG08ACA16Tx,16,cmr,0.96,45652.0
55,Seagate,Exos X16,ST12000NM001G,12,cmr,0.96,13147.0


## Data & API

Full dataset, always up to date: [nasdisks.com/data/](https://www.nasdisks.com/data/)

License: CC BY 4.0 for specs and CMR/SMR. Failure rates derived from [Backblaze Drive Stats](https://www.backblaze.com/cloud-storage/resources/hard-drive-test-data) - credit Backblaze when using.